In [1]:
!pip install -q "numpy<2.0" "protobuf~=3.20.0" ultralytics==8.3.221

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm
import yaml
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# ---------------- CONFIG ----------------
# Thư mục YOLO dataset đã chuẩn bị (images/train, images/val, labels/...)
DATA_ROOT = "/kaggle/input/yolo-data-ft/yolo_dataset_doan"

# File YAML gốc trong DATA_ROOT
YAML_IN  = os.path.join(DATA_ROOT, "dataset.yaml")
# File YAML fixed path dùng để train (trong /kaggle/working)
YAML_OUT = "/kaggle/working/dataset_fixed.yaml"

# Tên experiment YOLO (sẽ tạo folder /kaggle/working/runs/detect/<EXP_NAME>/)
EXP_NAME = "drone_real_synth"

# ----------------------------------------

In [ ]:
# ============================================================
# 0) CẤU HÌNH KIẾN TRÚC YOLO-DRONE (GHOSTHEAD)
# ============================================================
def create_yolo_drone_yaml(file_path="yolo_drone.yaml"):
    """
    Tạo file cấu hình kiến trúc YOLO-Drone dựa trên YOLOv11.
    Thay đổi chính ở phần HEAD:
    - Sử dụng C2f thay vì C3k2.
    - Sử dụng GhostConv thay vì Conv để downsample.
    """
    yolo_drone_cfg = """
# Ultralytics YOLO 🚀, AGPL-3.0 license
# YOLO-Drone architecture based on YOLOv11 with GhostHead
# Parameters
nc: 80 # number of classes (sẽ được override khi train)
scales: # model compound scaling constants, i.e. 'n'
  n: [0.50, 0.25, 1024] # summary: 319 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs

# YOLO11n backbone
backbone:
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

# YOLO-Drone GhostHead
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C2f, [512]] # 13 (Thay C3k2 bằng C2f)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C2f, [256]] # 16 (Thay C3k2 bằng C2f - Detect P3)

  - [-1, 1, GhostConv, [256, 3, 2]] # 17 (Thay Conv bằng GhostConv - Downsample)
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C2f, [512]] # 19 (Thay C3k2 bằng C2f - Detect P4)

  - [-1, 1, GhostConv, [512, 3, 2]] # 20 (Thay Conv bằng GhostConv - Downsample)
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 2, C2f, [1024]] # 22 (Thay C3k2 bằng C2f - Detect P5)

  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)
    """
    with open(file_path, "w") as f:
        f.write(yolo_drone_cfg.strip())
    print(f">> Đã tạo file kiến trúc YOLO-Drone: {file_path}")
    return file_path

# ============================================================
# 1) FIX dataset.yaml path (trỏ đúng đến DATA_ROOT)
# ============================================================

if 'YAML_IN' in locals() and 'DATA_ROOT' in locals():
    print(">> Đọc dataset.yaml gốc từ:", YAML_IN)
    with open(YAML_IN, "r") as f:
        cfg = yaml.safe_load(f)

    cfg["train"] = os.path.join(DATA_ROOT, "images/train")
    cfg["val"]   = os.path.join(DATA_ROOT, "images/val")

    print("New train path:", cfg["train"])
    print("New val   path:", cfg["val"])

    with open(YAML_OUT, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print(">> Đã ghi file YAML dataset mới:", YAML_OUT)


# ============================================================
# 2) Patch imread để tránh lỗi imdecode trong Ultralytics
# ============================================================
def patch_imread():
    import ultralytics.utils.patches as u_patches
    import ultralytics.data.base as u_base

    def my_imread(path, flags=cv2.IMREAD_COLOR):
        img = cv2.imread(str(path), flags)
        if img is None:
            print(f"[WARN] imread failed for: {path} -> using dummy image")
            img = np.zeros((640, 640, 3), dtype=np.uint8)
        return img

    u_patches.imread = my_imread
    u_base.imread    = my_imread
    print("[PATCH] Overrode ultralytics imread -> cv2.imread() + dummy fallback")


# ============================================================
# 3) TRAIN YOLO-DRONE (Architecture Modified)
# ============================================================
def train_yolo():
    """
    Train YOLO-Drone (GhostHead + C2f) trên dataset.
    Sử dụng Transfer Learning từ weights yolo11n.pt cho phần Backbone.
    """
    patch_imread()

    # Bước 1: Tạo file cấu hình kiến trúc mới
    yaml_model_path = create_yolo_drone_yaml("yolo_drone.yaml")

    # Bước 2: Khởi tạo model từ file YAML (để lấy kiến trúc mới)
    print(f"[INFO] Khởi tạo model từ kiến trúc: {yaml_model_path}")
    model = YOLO(yaml_model_path)

    # Bước 3: Load weights pretrained (Transfer Learning)
    # Ultralytics sẽ tự động load weights cho các layer trùng tên (Backbone)
    # và bỏ qua các layer khác biệt (Head mới)
    try:
        print("[INFO] Đang load weights pre-trained từ yolo11n.pt...")
        model = model.load("yolo11n.pt") 
    except Exception as e:
        print(f"[WARN] Không thể load weights trực tiếp (có thể do sai lệch shape): {e}")
        print("Model sẽ train từ đầu (Scratch) hoặc tiếp tục với weight random init.")

    # Bước 4: Train
    results = model.train(
        data=YAML_OUT,
        epochs=70,
        imgsz=640,
        batch=16,
        lr0=0.003,
        lrf=0.01,
        workers=2,
        amp=True,
        mixup=0.05,
        degrees=5.0,
        shear=2.0,
        name="yolo_drone_ghosthead" # Đổi tên experiment
    )
    # Cập nhật EXP_NAME để dùng cho inference
    global EXP_NAME
    EXP_NAME = "yolo_drone_ghosthead" 
    
    print(f"[YOLO] Train xong. Weights dir: /kaggle/working/runs/detect/{EXP_NAME}/weights/")
    return EXP_NAME

# ============================================================
# 5) MAIN
# ============================================================
if __name__ == "__main__":
    # Đảm bảo các biến toàn cục cần thiết đã được định nghĩa 
    # (YAML_IN, YAML_OUT, DATA_ROOT, EXP_NAME...) ở cell trước
    
    print("=== TRAIN YOLO-DRONE (GHOSTHEAD + C2F) ===")
    exp_name = train_yolo()

>> Đọc dataset.yaml gốc từ: /kaggle/input/yolo-data-ft/yolo_dataset_doan/dataset.yaml
New train path: /kaggle/input/yolo-data-ft/yolo_dataset_doan/images/train
New val   path: /kaggle/input/yolo-data-ft/yolo_dataset_doan/images/val
>> Đã ghi file YAML dataset mới: /kaggle/working/dataset_fixed.yaml
=== TRAIN YOLO-DRONE (GHOSTHEAD + C2F) ===
[PATCH] Overrode ultralytics imread -> cv2.imread() + dummy fallback
>> Đã tạo file kiến trúc YOLO-Drone: yolo_drone.yaml
[INFO] Khởi tạo model từ kiến trúc: yolo_drone.yaml
WARNING ⚠️ no model scale passed. Assuming scale='n'.
[INFO] Đang load weights pre-trained từ yolo11n.pt...
Transferred 429/481 items from pretrained weights
New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, cl

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       6041       6053      0.968      0.943      0.971      0.789
Speed: 0.1ms preprocess, 1.2ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /kaggle/working/runs/detect/yolo_drone_ghosthead
[YOLO] Train xong. Weights dir: /kaggle/working/runs/detect/yolo_drone_ghosthead/weights/
